## Retrieval Problem Definition

In [2]:
print("Customer features:", customer_features.shape)
print("Product features:", product_features.shape)
print("Order analytics:", order_analytics.shape)

print("\nCustomer columns:")
print(customer_features.columns.tolist())

print("\nProduct columns:")
print(product_features.columns.tolist())

print("\nOrder columns:")
print(order_analytics.columns.tolist())

Customer features: (4335, 13)
Product features: (3918, 11)
Order analytics: (19865, 9)

Customer columns:
['Customer ID', 'Total_Orders', 'Total_Revenue', 'Total_Units', 'Unique_Products', 'Recency_Days', 'Customer_Lifetime_Days', 'Average_Order_Value', 'Average_Units_Per_Order', 'Average_Products_Per_Order', 'Orders_Per_Month', 'Revenue_Segment', 'Activity_Segment']

Product columns:
['StockCode', 'Product_Name', 'Total_Units_Sold', 'Total_Revenue', 'Orders', 'Customers', 'Average_Price', 'Revenue_Per_Order', 'Units_Per_Order', 'Customer_Penetration', 'Revenue_Segment']

Order columns:
['Invoice', 'Order_Revenue', 'Units', 'Unique_Products', 'Average_Item_Value', 'Customer_ID', 'Country', 'Order_Date', 'Basket_Size']


In [3]:
print("CUSTOMER SAMPLE")
display(customer_features.head(3))

print("\nPRODUCT SAMPLE")
display(product_features.head(3))

print("\nORDER SAMPLE")
display(order_analytics.head(3))

CUSTOMER SAMPLE


,Customer ID,Total_Orders,Total_Revenue,Total_Units,Unique_Products,Recency_Days,Customer_Lifetime_Days,Average_Order_Value,Average_Units_Per_Order,Average_Products_Per_Order,Orders_Per_Month,Revenue_Segment,Activity_Segment
0,12346.0,1,77183.60,74215,1,326,0,77183.600000,74215.000000,1.000000,1.000000,Very High,Inactive
1,12347.0,7,4310.00,2458,103,2,365,615.714286,351.142857,14.714286,0.531646,Very High,Active
2,12348.0,4,1797.24,2341,22,75,282,449.310000,585.250000,5.500000,0.384615,Very High,Recently Inactive



PRODUCT SAMPLE


,StockCode,Product_Name,Total_Units_Sold,Total_Revenue,Orders,Customers,Average_Price,Revenue_Per_Order,Units_Per_Order,Customer_Penetration,Revenue_Segment
0,10002,INFLATABLE POLITICAL GLOBE,860,759.89,71,40,1.086620,10.702676,12.112676,0.922722,High
1,10080,GROOVY CACTUS INFLATABLE,303,119.09,22,19,0.410909,5.413182,13.772727,0.438293,Low
2,10120,DOGGY RUBBER,192,40.32,29,25,0.210000,1.390345,6.620690,0.576701,Low



ORDER SAMPLE


,Invoice,Order_Revenue,Units,Unique_Products,Average_Item_Value,Customer_ID,Country,Order_Date,Basket_Size
0,536365,139.12,40,7,3.478000,17850.0,United Kingdom,2010-12-01 08:26:00,Large
1,536366,22.20,12,2,1.850000,17850.0,United Kingdom,2010-12-01 08:28:00,Medium
2,536367,278.73,83,12,3.358193,13047.0,United Kingdom,2010-12-01 08:34:00,Very Large


In [6]:
retrieval_sources = {
    "customer": len(customer_features),
    "product": len(product_features),
    "order": len(order_analytics)
}

print("Retrieval sources:")
for source, count in retrieval_sources.items():
    print(f"- {source}: {count:,} records")

Retrieval sources:
- customer: 4,335 records
- product: 3,918 records
- order: 19,865 records


In [7]:
retrieval_questions = [
    "Which customers are high value?",
    "Which products generate the most revenue?",
    "What are the typical order characteristics?",
    "Which customers may need retention attention?",
    "What are the strongest product revenue contributors?"
]

print("Example retrieval questions:\n")

for i, question in enumerate(retrieval_questions, 1):
    print(f"{i}. {question}")

Example retrieval questions:

1. Which customers are high value?
2. Which products generate the most revenue?
3. What are the typical order characteristics?
4. Which customers may need retention attention?
5. What are the strongest product revenue contributors?


## CommerceIQ Knowledge Documents

In [8]:
customer_documents = []

for _, row in customer_features.iterrows():
    document = (
        f"Customer {row['Customer ID']} has "
        f"{int(row['Total_Orders'])} orders, "
        f"£{row['Total_Revenue']:,.2f} total revenue, "
        f"{int(row['Total_Units']):,} total units, "
        f"{int(row['Unique_Products'])} unique products, "
        f"an average order value of £{row['Average_Order_Value']:,.2f}, "
        f"revenue segment {row['Revenue_Segment']}, "
        f"and activity segment {row['Activity_Segment']}."
    )

    customer_documents.append({
        "document_id": f"customer_{row['Customer ID']}",
        "source_type": "customer",
        "text": document
    })

print("Customer documents:", len(customer_documents))
print("\nExample:")
print(customer_documents[0])

Customer documents: 4335

Example:
{'document_id': 'customer_12346.0', 'source_type': 'customer', 'text': 'Customer 12346.0 has 1 orders, £77,183.60 total revenue, 74,215 total units, 1 unique products, an average order value of £77,183.60, revenue segment Very High, and activity segment Inactive.'}


In [9]:
product_documents = []

for _, row in product_features.iterrows():
    document = (
        f"Product {row['Product_Name']} "
        f"(StockCode {row['StockCode']}) generated "
        f"£{row['Total_Revenue']:,.2f} revenue, "
        f"sold {int(row['Total_Units_Sold']):,} units, "
        f"appeared in {int(row['Orders']):,} orders, "
        f"reached {int(row['Customers']):,} customers, "
        f"has an average price of £{row['Average_Price']:.2f}, "
        f"and belongs to the {row['Revenue_Segment']} revenue segment."
    )

    product_documents.append({
        "document_id": f"product_{row['StockCode']}",
        "source_type": "product",
        "text": document
    })

print("Product documents:", len(product_documents))
print("\nExample:")
print(product_documents[0])

Product documents: 3918

Example:
{'document_id': 'product_10002', 'source_type': 'product', 'text': 'Product INFLATABLE POLITICAL GLOBE  (StockCode 10002) generated £759.89 revenue, sold 860 units, appeared in 71 orders, reached 40 customers, has an average price of £1.09, and belongs to the High revenue segment.'}


In [10]:
order_documents = []

for _, row in order_analytics.iterrows():
    document = (
        f"Order {row['Invoice']} from {row['Country']} "
        f"generated £{row['Order_Revenue']:,.2f} revenue, "
        f"contained {int(row['Units']):,} units, "
        f"{int(row['Unique_Products'])} unique products, "
        f"had an average item value of £{row['Average_Item_Value']:.2f}, "
        f"and had a {row['Basket_Size']} basket size."
    )

    order_documents.append({
        "document_id": f"order_{row['Invoice']}",
        "source_type": "order",
        "text": document
    })

print("Order documents:", len(order_documents))
print("\nExample:")
print(order_documents[0])

Order documents: 19865

Example:
{'document_id': 'order_536365', 'source_type': 'order', 'text': 'Order 536365 from United Kingdom generated £139.12 revenue, contained 40 units, 7 unique products, had an average item value of £3.48, and had a Large basket size.'}


In [11]:
documents = customer_documents + product_documents + order_documents

print("Total documents:", len(documents))

source_counts = pd.Series(
    [doc["source_type"] for doc in documents]
).value_counts()

print("\nDocuments by source:")
print(source_counts)

Total documents: 28118

Documents by source:
order       19865
customer     4335
product      3918
Name: count, dtype: int64


In [12]:
document_lengths = pd.Series(
    [len(doc["text"].split()) for doc in documents]
)

print("Total documents:", len(documents))
print("Average words:", document_lengths.mean())
print("Median words:", document_lengths.median())
print("Minimum words:", document_lengths.min())
print("Maximum words:", document_lengths.max())

print("\nEmpty documents:", sum(
    1 for doc in documents if not doc["text"].strip()
))

Total documents: 28118
Average words: 28.538765203784052
Median words: 28.0
Minimum words: 26
Maximum words: 38

Empty documents: 0


## Document Chunking

In [13]:
import numpy as np

word_counts = np.array([
    len(doc["text"].split())
    for doc in documents
])

print("Total documents:", len(word_counts))
print("Mean words:", word_counts.mean())
print("Median words:", np.median(word_counts))
print("95th percentile:", np.percentile(word_counts, 95))
print("Maximum words:", word_counts.max())

Total documents: 28118
Mean words: 28.538765203784052
Median words: 28.0
95th percentile: 34.0
Maximum words: 38


In [14]:
def chunk_text(text, chunk_size=50, overlap=10):
    words = text.split()
    chunks = []

    start = 0

    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))

        if end >= len(words):
            break

        start = end - overlap

    return chunks

In [15]:
test_text = documents[0]["text"]

test_chunks = chunk_text(
    test_text,
    chunk_size=20,
    overlap=5
)

print("Original document:")
print(test_text)

print("\nChunks:")
for i, chunk in enumerate(test_chunks, 1):
    print(f"\nChunk {i}:")
    print(chunk)

Original document:
Customer 12346.0 has 1 orders, £77,183.60 total revenue, 74,215 total units, 1 unique products, an average order value of £77,183.60, revenue segment Very High, and activity segment Inactive.

Chunks:

Chunk 1:
Customer 12346.0 has 1 orders, £77,183.60 total revenue, 74,215 total units, 1 unique products, an average order value of £77,183.60,

Chunk 2:
average order value of £77,183.60, revenue segment Very High, and activity segment Inactive.


In [16]:
chunked_documents = []

for doc in documents:
    chunks = chunk_text(
        doc["text"],
        chunk_size=50,
        overlap=10
    )

    for i, chunk in enumerate(chunks):
        chunked_documents.append({
            "chunk_id": f"{doc['document_id']}_chunk_{i}",
            "document_id": doc["document_id"],
            "source_type": doc["source_type"],
            "text": chunk
        })

print("Original documents:", len(documents))
print("Total chunks:", len(chunked_documents))

Original documents: 28118
Total chunks: 28118


In [17]:
chunk_lengths = pd.Series([
    len(chunk["text"].split())
    for chunk in chunked_documents
])

print("Total chunks:", len(chunked_documents))
print("Average words per chunk:", chunk_lengths.mean())
print("Median words per chunk:", chunk_lengths.median())
print("Minimum words:", chunk_lengths.min())
print("Maximum words:", chunk_lengths.max())

print("\nEmpty chunks:", sum(
    1 for chunk in chunked_documents
    if not chunk["text"].strip()
))

Total chunks: 28118
Average words per chunk: 28.538765203784052
Median words per chunk: 28.0
Minimum words: 26
Maximum words: 38

Empty chunks: 0


## Embedding Generation

In [20]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded.")
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

e:\commerceiq\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sadiya Sajid\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.
Embedding dimension: 384


C:\Users\Sadiya Sajid\AppData\Local\Temp\ipykernel_45528\186269946.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


In [21]:
chunk_texts = [
    chunk["text"]
    for chunk in chunked_documents
]

print("Texts to embed:", len(chunk_texts))
print("Example:")
print(chunk_texts[0])

Texts to embed: 28118
Example:
Customer 12346.0 has 1 orders, £77,183.60 total revenue, 74,215 total units, 1 unique products, an average order value of £77,183.60, revenue segment Very High, and activity segment Inactive.


In [22]:
embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)

Batches:   0%|          | 0/879 [00:00<?, ?it/s]

Embedding shape: (28118, 384)
Embedding dtype: float32


In [23]:
print("NaN values:", np.isnan(embeddings).sum())
print("Infinite values:", np.isinf(embeddings).sum())

embedding_norms = np.linalg.norm(embeddings, axis=1)

print("Minimum norm:", embedding_norms.min())
print("Maximum norm:", embedding_norms.max())
print("Average norm:", embedding_norms.mean())

NaN values: 0
Infinite values: 0
Minimum norm: 0.9999999
Maximum norm: 1.0000001
Average norm: 1.0


In [24]:
embeddings_path = "../data/commerceiq_embeddings.npy"
metadata_path = "../data/commerceiq_document_metadata.csv"

np.save(
    embeddings_path,
    embeddings
)

metadata = pd.DataFrame(chunked_documents)

metadata.to_csv(
    metadata_path,
    index=False
)

print("Saved embeddings:", embeddings_path)
print("Saved metadata:", metadata_path)
print("Metadata shape:", metadata.shape)

Saved embeddings: ../data/commerceiq_embeddings.npy
Saved metadata: ../data/commerceiq_document_metadata.csv
Metadata shape: (28118, 4)


## Vector Database

In [25]:
!pip install faiss-cpu

   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   ---- ----------------------------------- 1.8/16.2 MB 16.7 MB/s eta 0:00:01
   -------------- ------------------------- 6.0/16.2 MB 19.4 MB/s eta 0:00:01
   ------------------------ --------------- 10.0/16.2 MB 20.0 MB/s eta 0:00:01
   -------------------------------- ------- 13.4/16.2 MB 18.6 MB/s eta 0:00:01
   ------------------------------------ --- 14.9/16.2 MB 16.2 MB/s eta 0:00:01
   -------------------------------------- - 15.7/16.2 MB 13.7 MB/s eta 0:00:01
   ---------------------------------------- 16.2/16.2 MB 12.3 MB/s  0:00:01


In [26]:
import faiss
import numpy as np
import pandas as pd

embeddings = np.load("../data/commerceiq_embeddings.npy")
metadata = pd.read_csv("../data/commerceiq_document_metadata.csv")

print("Embeddings:", embeddings.shape)
print("Metadata:", metadata.shape)

Embeddings: (28118, 384)
Metadata: (28118, 4)


In [27]:
embedding_dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dimension)

index.add(embeddings)

print("Embedding dimension:", embedding_dimension)
print("Vectors in index:", index.ntotal)

Embedding dimension: 384
Vectors in index: 28118


In [28]:
print("Expected vectors:", len(metadata))
print("Indexed vectors:", index.ntotal)
print("Match:", index.ntotal == len(metadata))

Expected vectors: 28118
Indexed vectors: 28118
Match: True


In [29]:
query_vector = embeddings[0].reshape(1, -1)

scores, indices = index.search(
    query_vector,
    5
)

print("Retrieved indices:")
print(indices[0])

print("\nSimilarity scores:")
print(scores[0])

print("\nRetrieved documents:")

for idx, score in zip(indices[0], scores[0]):
    print(f"\nScore: {score:.4f}")
    print(metadata.iloc[idx]["text"])

Retrieved indices:
[ 0 10  8  5  1]

Similarity scores:
[1.         0.9840289  0.98334146 0.98233616 0.9812548 ]

Retrieved documents:

Score: 1.0000
Customer 12346.0 has 1 orders, £77,183.60 total revenue, 74,215 total units, 1 unique products, an average order value of £77,183.60, revenue segment Very High, and activity segment Inactive.

Score: 0.9840
Customer 12357.0 has 1 orders, £6,207.67 total revenue, 2,708 total units, 131 unique products, an average order value of £6,207.67, revenue segment Very High, and activity segment Recently Inactive.

Score: 0.9833
Customer 12355.0 has 1 orders, £459.40 total revenue, 240 total units, 13 unique products, an average order value of £459.40, revenue segment Medium, and activity segment Inactive.

Score: 0.9823
Customer 12352.0 has 7 orders, £1,665.74 total revenue, 533 total units, 58 unique products, an average order value of £237.96, revenue segment Very High, and activity segment Recently Inactive.

Score: 0.9813
Customer 12347.0 has 7

In [30]:
index_path = "../data/commerceiq_faiss.index"

faiss.write_index(
    index,
    index_path
)

print("FAISS index saved:", index_path)

FAISS index saved: ../data/commerceiq_faiss.index


## Semantic Retrieval

In [31]:
def retrieve_documents(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (idx, score) in enumerate(
        zip(indices[0], scores[0]), 1
    ):
        results.append({
            "Rank": rank,
            "Score": float(score),
            "Source": metadata.iloc[idx]["source_type"],
            "Document_ID": metadata.iloc[idx]["document_id"],
            "Text": metadata.iloc[idx]["text"]
        })

    return pd.DataFrame(results)

In [32]:
query = "Which customers are high value?"

customer_results = retrieve_documents(
    query,
    top_k=5
)

display(customer_results)

,Rank,Score,Source,Document_ID,Text
0,1,0.605892,customer,customer_14079.0,"Customer 14079.0 has 1 orders, £375.10 total r..."
1,2,0.604153,customer,customer_13500.0,"Customer 13500.0 has 3 orders, £1,166.77 total..."
2,3,0.602597,customer,customer_14208.0,"Customer 14208.0 has 1 orders, £151.74 total r..."
3,4,0.602385,customer,customer_16516.0,"Customer 16516.0 has 1 orders, £101.70 total r..."
4,5,0.601726,customer,customer_15069.0,"Customer 15069.0 has 1 orders, £1,109.53 total..."


In [33]:
query = "Which products generate the most revenue?"

product_results = retrieve_documents(
    query,
    top_k=5
)

display(product_results)

,Rank,Score,Source,Document_ID,Text
0,1,0.520888,product,product_22443,Product GROW YOUR OWN HERBS SET OF 3 (StockCod...
1,2,0.518029,order,order_560359,Order 560359 from United Kingdom generated £33...
2,3,0.514124,order,order_566292,Order 566292 from United Kingdom generated £52...
3,4,0.510614,order,order_545859,Order 545859 from United Kingdom generated £59...
4,5,0.508282,order,order_570293,Order 570293 from United Kingdom generated £33...


In [34]:
query = "What are the typical order characteristics?"

order_results = retrieve_documents(
    query,
    top_k=5
)

display(order_results)

,Rank,Score,Source,Document_ID,Text
0,1,0.365007,customer,customer_17440.0,"Customer 17440.0 has 3 orders, £186.42 total r..."
1,2,0.359393,customer,customer_17883.0,"Customer 17883.0 has 4 orders, £664.34 total r..."
2,3,0.356294,customer,customer_17813.0,"Customer 17813.0 has 5 orders, £1,698.07 total..."
3,4,0.355938,customer,customer_18273.0,"Customer 18273.0 has 3 orders, £204.00 total r..."
4,5,0.353427,customer,customer_17893.0,"Customer 17893.0 has 1 orders, £107.21 total r..."


In [35]:
query = "Which customers may need retention attention?"

retention_results = retrieve_documents(
    query,
    top_k=5
)

display(retention_results)

,Rank,Score,Source,Document_ID,Text
0,1,0.400068,customer,customer_14073.0,"Customer 14073.0 has 1 orders, £151.04 total r..."
1,2,0.396603,customer,customer_16221.0,"Customer 16221.0 has 3 orders, £860.86 total r..."
2,3,0.391590,customer,customer_16337.0,"Customer 16337.0 has 1 orders, £151.05 total r..."
3,4,0.390952,customer,customer_15937.0,"Customer 15937.0 has 1 orders, £145.35 total r..."
4,5,0.390593,customer,customer_13462.0,"Customer 13462.0 has 1 orders, £151.50 total r..."


In [36]:
test_results = pd.concat([
    customer_results,
    product_results,
    order_results,
    retention_results
])

print(
    test_results.groupby("Source").size()
)

print("\nAverage similarity by source:")
print(
    test_results.groupby("Source")["Score"].mean()
)

Source
customer    15
order        4
product      1
dtype: int64

Average similarity by source:
Source
customer    0.451775
order       0.512762
product     0.520888
Name: Score, dtype: float64


## Retrieval Optimization

In [37]:
business_documents = []

# Customer revenue summary
customer_revenue_summary = (
    customer_features
    .groupby("Revenue_Segment")
    .agg(
        Customers=("Customer ID", "count"),
        Total_Revenue=("Total_Revenue", "sum"),
        Average_Revenue=("Total_Revenue", "mean")
    )
    .sort_values("Total_Revenue", ascending=False)
)

for segment, row in customer_revenue_summary.iterrows():
    text = (
        f"Customer revenue segment {segment} contains "
        f"{int(row['Customers']):,} customers and generates "
        f"£{row['Total_Revenue']:,.2f} total revenue, "
        f"with average revenue of £{row['Average_Revenue']:,.2f} per customer."
    )

    business_documents.append({
        "document_id": f"customer_segment_{segment}",
        "source_type": "customer_segment",
        "text": text
    })


# Product revenue summary
top_products = (
    product_features
    .sort_values("Total_Revenue", ascending=False)
    .head(20)
)

for _, row in top_products.iterrows():
    text = (
        f"Product revenue analysis: {row['Product_Name']} "
        f"generated £{row['Total_Revenue']:,.2f} revenue, "
        f"sold {int(row['Total_Units_Sold']):,} units, "
        f"and appeared in {int(row['Orders']):,} orders."
    )

    business_documents.append({
        "document_id": f"product_summary_{row['StockCode']}",
        "source_type": "product_summary",
        "text": text
    })


print("Business summary documents:", len(business_documents))
print("\nExamples:")

for doc in business_documents[:3]:
    print(doc)

Business summary documents: 24

Examples:
{'document_id': 'customer_segment_Very High', 'source_type': 'customer_segment', 'text': 'Customer revenue segment Very High contains 1,084 customers and generates £6,988,949.30 total revenue, with average revenue of £6,447.37 per customer.'}
{'document_id': 'customer_segment_High', 'source_type': 'customer_segment', 'text': 'Customer revenue segment High contains 1,083 customers and generates £1,152,263.23 total revenue, with average revenue of £1,063.95 per customer.'}
{'document_id': 'customer_segment_Medium', 'source_type': 'customer_segment', 'text': 'Customer revenue segment Medium contains 1,084 customers and generates £500,073.49 total revenue, with average revenue of £461.32 per customer.'}


In [38]:
retention_summary = (
    customer_features
    .groupby("Activity_Segment")
    .agg(
        Customers=("Customer ID", "count"),
        Total_Revenue=("Total_Revenue", "sum"),
        Average_Revenue=("Total_Revenue", "mean"),
        Average_Recency=("Recency_Days", "mean")
    )
    .sort_values("Total_Revenue", ascending=False)
)

retention_documents = []

for segment, row in retention_summary.iterrows():
    text = (
        f"Customer activity segment {segment} contains "
        f"{int(row['Customers']):,} customers, "
        f"generates £{row['Total_Revenue']:,.2f} total revenue, "
        f"has average revenue of £{row['Average_Revenue']:,.2f}, "
        f"and average recency of {row['Average_Recency']:.1f} days."
    )

    retention_documents.append({
        "document_id": f"activity_{segment}",
        "source_type": "activity_summary",
        "text": text
    })

print("Retention documents:", len(retention_documents))

for doc in retention_documents:
    print("\n", doc)

Retention documents: 4

 {'document_id': 'activity_Active', 'source_type': 'activity_summary', 'text': 'Customer activity segment Active contains 1,647 customers, generates £6,224,125.47 total revenue, has average revenue of £3,779.07, and average recency of 13.5 days.'}

 {'document_id': 'activity_Recently Inactive', 'source_type': 'activity_summary', 'text': 'Customer activity segment Recently Inactive contains 1,240 customers, generates £1,584,214.25 total revenue, has average revenue of £1,277.59, and average recency of 56.2 days.'}

 {'document_id': 'activity_Inactive', 'source_type': 'activity_summary', 'text': 'Customer activity segment Inactive contains 862 customers, generates £558,220.03 total revenue, has average revenue of £647.59, and average recency of 269.3 days.'}

 {'document_id': 'activity_At Risk', 'source_type': 'activity_summary', 'text': 'Customer activity segment At Risk contains 586 customers, generates £467,247.21 total revenue, has average revenue of £797.35, 

In [39]:
all_retrieval_documents = (
    documents
    + business_documents
    + retention_documents
)

print("Original documents:", len(documents))
print("Business summary documents:", len(business_documents))
print("Retention documents:", len(retention_documents))
print("Total retrieval documents:", len(all_retrieval_documents))

Original documents: 28118
Business summary documents: 24
Retention documents: 4
Total retrieval documents: 28146


In [40]:
summary_texts = [
    doc["text"]
    for doc in business_documents + retention_documents
]

summary_embeddings = embedding_model.encode(
    summary_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Summary embeddings:", summary_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Summary embeddings: (28, 384)


In [41]:
all_texts = [
    doc["text"]
    for doc in all_retrieval_documents
]

all_embeddings = np.vstack([
    embeddings,
    summary_embeddings
])

retrieval_metadata = pd.DataFrame(all_retrieval_documents)

print("Total embeddings:", all_embeddings.shape)
print("Metadata:", retrieval_metadata.shape)

Total embeddings: (28146, 384)
Metadata: (28146, 3)


In [42]:
improved_index = faiss.IndexFlatIP(
    all_embeddings.shape[1]
)

improved_index.add(all_embeddings)

print("Vectors in improved index:", improved_index.ntotal)
print("Metadata records:", len(retrieval_metadata))
print("Match:", improved_index.ntotal == len(retrieval_metadata))

Vectors in improved index: 28146
Metadata records: 28146
Match: True


In [43]:
def retrieve_improved(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    scores, indices = improved_index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (idx, score) in enumerate(
        zip(indices[0], scores[0]), 1
    ):
        results.append({
            "Rank": rank,
            "Score": float(score),
            "Source": retrieval_metadata.iloc[idx]["source_type"],
            "Document_ID": retrieval_metadata.iloc[idx]["document_id"],
            "Text": retrieval_metadata.iloc[idx]["text"]
        })

    return pd.DataFrame(results)

In [44]:
product_test = retrieve_improved(
    "Which products generate the most revenue?",
    top_k=5
)

display(product_test)

,Rank,Score,Source,Document_ID,Text
0,1,0.520888,product,product_22443,Product GROW YOUR OWN HERBS SET OF 3 (StockCod...
1,2,0.518029,order,order_560359,Order 560359 from United Kingdom generated £33...
2,3,0.514124,order,order_566292,Order 566292 from United Kingdom generated £52...
3,4,0.510614,order,order_545859,Order 545859 from United Kingdom generated £59...
4,5,0.508282,order,order_570293,Order 570293 from United Kingdom generated £33...


In [45]:
customer_test = retrieve_improved(
    "Which customer revenue segment generates the most revenue?",
    top_k=5
)

display(customer_test)

,Rank,Score,Source,Document_ID,Text
0,1,0.769266,customer_segment,customer_segment_Medium,"Customer revenue segment Medium contains 1,084..."
1,2,0.766475,customer_segment,customer_segment_High,"Customer revenue segment High contains 1,083 c..."
2,3,0.745968,customer_segment,customer_segment_Very High,"Customer revenue segment Very High contains 1,..."
3,4,0.636120,customer_segment,customer_segment_Low,"Customer revenue segment Low contains 1,084 cu..."
4,5,0.624922,activity_summary,activity_Active,"Customer activity segment Active contains 1,64..."


In [46]:
retention_test = retrieve_improved(
    "Which customer groups may need retention attention?",
    top_k=5
)

display(retention_test)

,Rank,Score,Source,Document_ID,Text
0,1,0.397930,activity_summary,activity_Inactive,Customer activity segment Inactive contains 86...
1,2,0.395691,activity_summary,activity_Recently Inactive,Customer activity segment Recently Inactive co...
2,3,0.375049,customer,customer_16221.0,"Customer 16221.0 has 3 orders, £860.86 total r..."
3,4,0.364917,customer,customer_14073.0,"Customer 14073.0 has 1 orders, £151.04 total r..."
4,5,0.363133,activity_summary,activity_Active,"Customer activity segment Active contains 1,64..."


## Source-Aware Retrieval

In [47]:

customer_segment_docs = [
    doc for doc in all_retrieval_documents
    if doc["source_type"] == "customer_segment"
]

product_summary_docs = [
    doc for doc in all_retrieval_documents
    if doc["source_type"] == "product_summary"
]

activity_summary_docs = [
    doc for doc in all_retrieval_documents
    if doc["source_type"] == "activity_summary"
]

print("Customer segment documents:", len(customer_segment_docs))
print("Product summary documents:", len(product_summary_docs))
print("Activity summary documents:", len(activity_summary_docs))

Customer segment documents: 4
Product summary documents: 20
Activity summary documents: 4


In [48]:
# Build a small index for product summaries

product_summary_texts = [doc["text"] for doc in product_summary_docs]

product_summary_embeddings = embedding_model.encode(
    product_summary_texts,
    batch_size=32,
    show_progress_bar=False,
    normalize_embeddings=True
)

product_index = faiss.IndexFlatIP(product_summary_embeddings.shape[1])
product_index.add(product_summary_embeddings)

print("Product summary vectors:", product_index.ntotal)

Product summary vectors: 20


In [49]:
def retrieve_product_summaries(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    scores, indices = product_index.search(query_embedding, top_k)

    results = []

    for rank, (idx, score) in enumerate(
        zip(indices[0], scores[0]), 1
    ):
        doc = product_summary_docs[idx]

        results.append({
            "Rank": rank,
            "Score": float(score),
            "Source": doc["source_type"],
            "Document_ID": doc["document_id"],
            "Text": doc["text"]
        })

    return pd.DataFrame(results)

In [50]:
product_results = retrieve_product_summaries(
    "Which products generate the most revenue?",
    top_k=5
)

product_results

,Rank,Score,Source,Document_ID,Text
0,1,0.492636,product_summary,product_summary_22197,Product revenue analysis: SMALL POPCORN HOLDER...
1,2,0.473959,product_summary,product_summary_23166,Product revenue analysis: MEDIUM CERAMIC TOP S...
2,3,0.471828,product_summary,product_summary_POST,Product revenue analysis: POSTAGE generated £7...
3,4,0.460692,product_summary,product_summary_DOT,Product revenue analysis: DOTCOM POSTAGE gener...
4,5,0.447714,product_summary,product_summary_84879,Product revenue analysis: ASSORTED COLOUR BIRD...


In [51]:
customer_results = retrieve_improved(
    "Which customer revenue segment generates the most revenue?",
    top_k=5
)

customer_results

,Rank,Score,Source,Document_ID,Text
0,1,0.769266,customer_segment,customer_segment_Medium,"Customer revenue segment Medium contains 1,084..."
1,2,0.766475,customer_segment,customer_segment_High,"Customer revenue segment High contains 1,083 c..."
2,3,0.745968,customer_segment,customer_segment_Very High,"Customer revenue segment Very High contains 1,..."
3,4,0.636120,customer_segment,customer_segment_Low,"Customer revenue segment Low contains 1,084 cu..."
4,5,0.624922,activity_summary,activity_Active,"Customer activity segment Active contains 1,64..."


In [52]:
retention_results = retrieve_improved(
    "Which customer groups may need retention attention?",
    top_k=5
)

retention_results

,Rank,Score,Source,Document_ID,Text
0,1,0.397930,activity_summary,activity_Inactive,Customer activity segment Inactive contains 86...
1,2,0.395691,activity_summary,activity_Recently Inactive,Customer activity segment Recently Inactive co...
2,3,0.375049,customer,customer_16221.0,"Customer 16221.0 has 3 orders, £860.86 total r..."
3,4,0.364917,customer,customer_14073.0,"Customer 14073.0 has 1 orders, £151.04 total r..."
4,5,0.363133,activity_summary,activity_Active,"Customer activity segment Active contains 1,64..."


## Retrieval + Business Ranking

In [54]:
excluded_keywords = [
    "POSTAGE",
    "DOTCOM POSTAGE",
    "BANK CHARGES",
    "SAMPLES",
    "BAD DEBT",
    "ADJUSTMENT",
    "MANUAL"
]

product_features_clean = product_features.copy()

product_features_clean["Product_Name_Upper"] = (
    product_features_clean["Product_Name"]
    .fillna("")
    .str.upper()
)

merchandise_products = product_features_clean[
    ~product_features_clean["Product_Name_Upper"].apply(
        lambda x: any(keyword in x for keyword in excluded_keywords)
    )
].copy()

merchandise_products = merchandise_products.sort_values(
    "Total_Revenue",
    ascending=False
)

print("Total products:", len(product_features_clean))
print("Merchandise products:", len(merchandise_products))

merchandise_products[
    ["StockCode", "Product_Name", "Total_Revenue"]
].head(10)

Total products: 3918
Merchandise products: 3914


,StockCode,Product_Name,Total_Revenue
1310,22423,REGENCY CAKESTAND 3 TIER,174156.54
2465,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60
3407,85123A,WHITE HANGING HEART T-LIGHT HOLDER,104462.75
2670,47566,PARTY BUNTING,99445.23
3387,85099B,JUMBO BAG RED RETROSPOT,94159.81
2020,23166,MEDIUM CERAMIC TOP STORAGE JAR,81700.92
1942,23084,RABBIT NIGHT LIGHT,66870.03
1006,22086,PAPER CHAIN KIT 50'S CHRISTMAS,64875.59
3194,84879,ASSORTED COLOUR BIRD ORNAMENT,58927.62
2845,79321,CHILLI LIGHTS,54096.36


In [55]:
def rank_products_by_revenue(top_k=10):
    results = merchandise_products[
        [
            "StockCode",
            "Product_Name",
            "Total_Revenue",
            "Total_Units_Sold",
            "Orders",
            "Customers"
        ]
    ].head(top_k).copy()

    results.insert(0, "Rank", range(1, len(results) + 1))

    return results.reset_index(drop=True)


top_products = rank_products_by_revenue(10)

top_products

,Rank,StockCode,Product_Name,Total_Revenue,Total_Units_Sold,Orders,Customers
0,1,22423,REGENCY CAKESTAND 3 TIER,174156.54,13851,1988,881
1,2,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1,1
2,3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,104462.75,37641,2198,856
3,4,47566,PARTY BUNTING,99445.23,18283,1685,708
4,5,85099B,JUMBO BAG RED RETROSPOT,94159.81,48371,2089,635
5,6,23166,MEDIUM CERAMIC TOP STORAGE JAR,81700.92,78033,247,138
6,7,23084,RABBIT NIGHT LIGHT,66870.03,30739,994,450
7,8,22086,PAPER CHAIN KIT 50'S CHRISTMAS,64875.59,19329,1160,613
8,9,84879,ASSORTED COLOUR BIRD ORNAMENT,58927.62,36362,1455,678
9,10,79321,CHILLI LIGHTS,54096.36,10302,661,205


In [56]:
def retrieve_and_rank_products(query, retrieval_k=10, top_k=5):
    # Semantic retrieval
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    scores, indices = product_index.search(
        query_embedding,
        retrieval_k
    )

    retrieved_ids = [
        product_summary_docs[idx]["document_id"]
        for idx in indices[0]
    ]

    # Convert retrieved stock codes back to product data
    retrieved_stock_codes = [
        doc_id.replace("product_summary_", "")
        for doc_id in retrieved_ids
    ]

    candidates = merchandise_products[
        merchandise_products["StockCode"].astype(str).isin(
            retrieved_stock_codes
        )
    ].copy()

    # Business ranking: highest revenue first
    candidates = candidates.sort_values(
        "Total_Revenue",
        ascending=False
    ).head(top_k)

    candidates.insert(
        0,
        "Rank",
        range(1, len(candidates) + 1)
    )

    return candidates[
        [
            "Rank",
            "StockCode",
            "Product_Name",
            "Total_Revenue",
            "Total_Units_Sold",
            "Orders",
            "Customers"
        ]
    ].reset_index(drop=True)

In [57]:
product_ranked_results = retrieve_and_rank_products(
    "Which products generate the most revenue?",
    retrieval_k=20,
    top_k=5
)

product_ranked_results

,Rank,StockCode,Product_Name,Total_Revenue,Total_Units_Sold,Orders,Customers
0,1,22423,REGENCY CAKESTAND 3 TIER,174156.54,13851,1988,881
1,2,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1,1
2,3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,104462.75,37641,2198,856
3,4,47566,PARTY BUNTING,99445.23,18283,1685,708
4,5,85099B,JUMBO BAG RED RETROSPOT,94159.81,48371,2089,635


In [58]:
# Compare semantic retrieval with actual global revenue ranking

global_top_5 = rank_products_by_revenue(5)

print("Global top 5 merchandise products:")
display(global_top_5)

print("\nRetrieved + business-ranked products:")
display(product_ranked_results)

Global top 5 merchandise products:


,Rank,StockCode,Product_Name,Total_Revenue,Total_Units_Sold,Orders,Customers
0,1,22423,REGENCY CAKESTAND 3 TIER,174156.54,13851,1988,881
1,2,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1,1
2,3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,104462.75,37641,2198,856
3,4,47566,PARTY BUNTING,99445.23,18283,1685,708
4,5,85099B,JUMBO BAG RED RETROSPOT,94159.81,48371,2089,635



Retrieved + business-ranked products:


,Rank,StockCode,Product_Name,Total_Revenue,Total_Units_Sold,Orders,Customers
0,1,22423,REGENCY CAKESTAND 3 TIER,174156.54,13851,1988,881
1,2,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1,1
2,3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,104462.75,37641,2198,856
3,4,47566,PARTY BUNTING,99445.23,18283,1685,708
4,5,85099B,JUMBO BAG RED RETROSPOT,94159.81,48371,2089,635
